# Lab 3 — NYC Taxi Data: Profile & Clean
**INF4058 Data Engineering**

Look → Profile → Find problems → Decide → Clean → **Stop**.
The cleaned data is **not** loaded into PostgreSQL in this lab.

In [23]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_FILE = "yellow_tripdata_2021-01.csv.gz"   # same folder as this notebook (not pushed to GitHub)
CHUNK_SIZE = 100_000
DATE_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

# Download the data file only if it is not already here (works in Jupyter and Google Colab)
import os, urllib.request
URL = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
if not os.path.exists(DATA_FILE):
    print("Downloading data file...")
    urllib.request.urlretrieve(URL, DATA_FILE)
print("Data file ready:", DATA_FILE)

Data file ready: yellow_tripdata_2021-01.csv.gz


## Step 1 — Look
Read only the first 100 rows to understand the structure.

In [24]:
df_sample = pd.read_csv(DATA_FILE, nrows=100)
print(df_sample.head())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132           165             1         42.0    0.5      0.5   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0          10.60           1                  N           138           132             1         29.0    0.5      0.5   
4         2  2021-01-01 00:31:49   2021-

In [25]:
print(df_sample.dtypes)

VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


**Observations**
- **One row = one taxi trip**: pickup/dropoff time, pickup/dropoff zone, distance, passengers and the fare breakdown.
- **Date columns:** `tpep_pickup_datetime` and `tpep_dropoff_datetime`. pandas reads them as text (`object`/`str`), not as datetimes, so they must be parsed.
- The sample has no empty columns, but the full-file profile below shows that `VendorID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag` and `payment_type` are empty in a block of rows.

## Step 2 — Profile the whole file (in chunks)
The file is read in chunks of 100,000 rows. Only small running totals are kept (row count, nulls, min, max per column),
so the full file is never held in memory. The data is not modified.

In [26]:
total_rows = 0
null_counts = None
col_min, col_max, col_dtypes = {}, {}, {}

for chunk in pd.read_csv(DATA_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    total_rows += len(chunk)

    nulls = chunk.isna().sum()
    null_counts = nulls if null_counts is None else null_counts + nulls

    for col in chunk.columns:
        col_dtypes.setdefault(col, set()).add(str(chunk[col].dtype))
        values = chunk[col].dropna()
        if values.empty:
            continue
        mn, mx = values.min(), values.max()
        col_min[col] = mn if col not in col_min else min(col_min[col], mn)
        col_max[col] = mx if col not in col_max else max(col_max[col], mx)

profile = pd.DataFrame({
    "dtype": {c: " / ".join(sorted(t)) for c, t in col_dtypes.items()},
    "nulls": null_counts,
    "null_%": (null_counts / total_rows * 100).round(2),
    "min": pd.Series(col_min),
    "max": pd.Series(col_max),
})

print(f"Total rows: {total_rows:,}\n")
print(profile)

Total rows: 1,369,765

                                 dtype  nulls  null_%                  min                  max
VendorID               float64 / int64  98352    7.18                    1                    2
tpep_pickup_datetime               str      0    0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime              str      0    0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count        float64 / int64  98352    7.18                    0                    8
trip_distance                  float64      0    0.00                  0.0            263163.28
RatecodeID             float64 / int64  98352    7.18                    1                   99
store_and_fwd_flag       float64 / str  98352    7.18                    N                    Y
PULocationID                     int64      0    0.00                    1                  265
DOLocationID                     int64      0    0.00                    1                  265
payment_type     

**What the profile shows**
- **1,369,765 rows** — same count as the Postgres table in class.
- **98,352 rows (7.18%)** have no `VendorID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag` or `payment_type`. These columns are empty together in the same rows, which is why they come out as `float64` instead of integers (this is why the class pipeline used `Int64`).
- Pickup dates start in **2008** and end in **February 2021**, although the file is for January 2021.
- `fare_amount`, `total_amount`, `tip_amount` and the surcharges have **negative minimums**.
- `trip_distance` goes from **0** up to **263,163 miles**.
- `passenger_count` min is **0**; `RatecodeID` max is **99** (not a valid code).

## Step 3 — Find problems
Each check is counted chunk by chunk. Exact duplicates are found across the whole file
by hashing every row and remembering the hashes already seen.

In [27]:
problems = {
    "pickup outside Jan 2021": 0,
    "dropoff before or equal to pickup": 0,
    "fare_amount <= 0": 0,
    "fare_amount < 0 (negative)": 0,
    "trip_distance <= 0": 0,
    "trip_distance > 100 miles": 0,
    "passenger_count == 0": 0,
    "passenger_count is null": 0,
    "duplicate rows": 0,
}
seen_hashes = set()

for chunk in pd.read_csv(DATA_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    pickup = pd.to_datetime(chunk["tpep_pickup_datetime"])
    dropoff = pd.to_datetime(chunk["tpep_dropoff_datetime"])

    problems["pickup outside Jan 2021"] += ((pickup < "2021-01-01") | (pickup >= "2021-02-01")).sum()
    problems["dropoff before or equal to pickup"] += (dropoff <= pickup).sum()
    problems["fare_amount <= 0"] += (chunk["fare_amount"] <= 0).sum()
    problems["fare_amount < 0 (negative)"] += (chunk["fare_amount"] < 0).sum()
    problems["trip_distance <= 0"] += (chunk["trip_distance"] <= 0).sum()
    problems["trip_distance > 100 miles"] += (chunk["trip_distance"] > 100).sum()
    problems["passenger_count == 0"] += (chunk["passenger_count"] == 0).sum()
    problems["passenger_count is null"] += chunk["passenger_count"].isna().sum()

    for h in pd.util.hash_pandas_object(chunk, index=False):
        if h in seen_hashes:
            problems["duplicate rows"] += 1
        else:
            seen_hashes.add(h)

problems_df = pd.DataFrame({"rows": pd.Series(problems).astype(int)})
problems_df["% of file"] = (problems_df["rows"] / total_rows * 100).round(3)
print(problems_df)

                                    rows  % of file
pickup outside Jan 2021               24      0.002
dropoff before or equal to pickup   6557      0.479
fare_amount <= 0                    7411      0.541
fare_amount < 0 (negative)          6769      0.494
trip_distance <= 0                 19952      1.457
trip_distance > 100 miles             55      0.004
passenger_count == 0               26726      1.951
passenger_count is null            98352      7.180
duplicate rows                         0      0.000


In [28]:
# Look at the bad pickup dates themselves
bad_dates = []
for chunk in pd.read_csv(DATA_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    p = pd.to_datetime(chunk["tpep_pickup_datetime"])
    bad_dates.append(chunk.loc[(p < "2021-01-01") | (p >= "2021-02-01"),
                               ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount"]])
print(pd.concat(bad_dates))

        tpep_pickup_datetime tpep_dropoff_datetime  trip_distance  fare_amount
455      2020-12-31 23:59:06   2021-01-01 00:02:51           0.74          5.0
530      2020-12-31 21:40:20   2020-12-31 22:16:09          17.96         49.5
546      2020-12-31 23:57:17   2021-01-01 00:17:40           3.80         16.0
637      2020-12-31 18:41:27   2021-01-01 17:52:03           2.61         10.0
902      2020-12-31 23:55:40   2021-01-01 00:24:51           6.45         24.5
936      2020-12-31 13:44:02   2020-12-31 13:53:43           3.76         12.5
1043     2020-12-31 23:59:40   2021-01-01 00:25:51          12.73         36.5
1661     2020-12-31 21:35:59   2020-12-31 21:38:20           0.66          4.0
1662     2020-12-31 21:43:40   2020-12-31 21:52:26           2.15          9.0
1880     2020-12-31 23:31:29   2020-12-31 23:37:51           1.57          7.5
2846     2020-12-31 18:25:31   2020-12-31 18:35:41           2.72         10.5
2847     2020-12-31 18:47:49   2021-01-01 17:54:51  

## Step 4 — Decide: drop, fix or flag
- **Drop** = the row is not a valid January 2021 trip, so it should not reach the table.
- **Flag** = the trip is real but one field is doubtful. Keep the row and add a boolean column, so the analyst decides.
- **Fix** = the value is wrong in a predictable way and can be corrected safely.

In [29]:
decisions = pd.DataFrame([
    ["Null passenger_count",              "drop", "98,352 rows miss vendor, passengers, rate code and payment type together. Without them the trip cannot be used for passenger or payment analysis.", "Data owner (TLC / vendor)"],
    ["Pickup outside Jan 2021",           "drop", "Belongs to another month or is a clock error (2008/2009 dates). Would double count trips when other months are loaded.", "Data engineering (pipeline rule)"],
    ["Dropoff before/equal to pickup",    "drop", "A trip cannot end before it starts; duration would be zero or negative.",                                       "Data engineering"],
    ["fare_amount <= 0",                  "drop", "Negative fares are refunds/voids and 0 fares are cancelled trips — not revenue trips.",                          "Finance / business team"],
    ["trip_distance <= 0",                "flag", "Fare is often valid (flat rate, meter not moved, GPS issue), so dropping would lose real revenue.",              "Analytics team"],
    ["trip_distance > 100 miles",         "flag", "Possible meter error (max is 263k miles) but could be a real long trip. Analyst decides.",                      "Analytics team"],
    ["passenger_count == 0",              "flag", "Driver did not enter the count; the trip itself is real.",                                                      "Data owner (TLC / vendor)"],
    ["Float type on integer columns",     "fix",  "Columns with nulls are read as float64; store them as nullable Int64 so IDs and counts stay integers.",          "Data engineering"],
    ["Duplicate rows",                    "drop", "An exact copy would count the same trip twice. Found 0, but the rule stays for future months.",                  "Data engineering"],
], columns=["problem", "decision", "why", "rule owner"])

print(decisions.to_string(index=False))

                       problem decision                                                                                                                                               why                       rule owner
          Null passenger_count     drop 98,352 rows miss vendor, passengers, rate code and payment type together. Without them the trip cannot be used for passenger or payment analysis.        Data owner (TLC / vendor)
       Pickup outside Jan 2021     drop                            Belongs to another month or is a clock error (2008/2009 dates). Would double count trips when other months are loaded. Data engineering (pipeline rule)
Dropoff before/equal to pickup     drop                                                                           A trip cannot end before it starts; duration would be zero or negative.                 Data engineering
              fare_amount <= 0     drop                                                             Negative fares are refun

## Step 5 — Clean
`clean(df)`:
- works on a **copy** — the input dataframe is never changed,
- applies the rules **in order** and counts how many rows each rule removes (nothing is dropped silently),
- adds `flag_*` columns instead of deleting doubtful rows,
- returns `(cleaned_df, {rule: rows_removed})`.

In [30]:
INT_COLS = ["VendorID", "passenger_count", "RatecodeID", "payment_type"]


def clean(df):
    out = df.copy()                      # never modify the original dataframe
    removed = {}

    def drop_rows(rule, bad_mask):
        nonlocal out
        removed[rule] = int(bad_mask.sum())
        out = out.loc[~bad_mask]

    # --- fix: types
    for col in DATE_COLS:
        out[col] = pd.to_datetime(out[col])
    for col in INT_COLS:
        out[col] = out[col].astype("Int64")   # nullable integer, keeps NaN as <NA>

    # --- drop rules (order matters: each counts rows still left after the previous ones)
    drop_rows("passenger_count_null",
              out["passenger_count"].isna())
    drop_rows("pickup_outside_jan_2021",
              (out["tpep_pickup_datetime"] < "2021-01-01") | (out["tpep_pickup_datetime"] >= "2021-02-01"))
    drop_rows("dropoff_not_after_pickup",
              out["tpep_dropoff_datetime"] <= out["tpep_pickup_datetime"])
    drop_rows("fare_amount_le_0",
              out["fare_amount"] <= 0)
    drop_rows("duplicate_rows",
              out.duplicated())

    # --- flag rules (rows kept, 0 removed)
    out["flag_zero_distance"] = out["trip_distance"] <= 0
    out["flag_long_distance"] = out["trip_distance"] > 100
    out["flag_zero_passengers"] = (out["passenger_count"] == 0).fillna(False).astype(bool)
    for rule in ["flag_zero_distance", "flag_long_distance", "flag_zero_passengers"]:
        removed[rule] = 0

    return out.reset_index(drop=True), removed

Run `clean()` on every chunk and add up the results.
Rows removed and rows flagged are added up across all 14 chunks.

In [31]:
rows_before = 0
rows_after = 0
removed_total = {}
flagged_total = {}
original_untouched = True

for chunk in pd.read_csv(DATA_FILE, chunksize=CHUNK_SIZE, low_memory=False):
    snapshot = chunk.copy()
    cleaned, removed = clean(chunk)
    original_untouched &= chunk.equals(snapshot)     # proof clean() did not change its input

    rows_before += len(chunk)
    rows_after += len(cleaned)
    for rule, n in removed.items():
        removed_total[rule] = removed_total.get(rule, 0) + n
    for col in [c for c in cleaned.columns if c.startswith("flag_")]:
        flagged_total[col] = flagged_total.get(col, 0) + int(cleaned[col].sum())

print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}\n")

print("{rule: rows_removed}")
print(removed_total, "\n")

summary = pd.DataFrame({"rows_removed": pd.Series(removed_total),
                        "rows_flagged (kept)": pd.Series(flagged_total)}).fillna(0).astype(int)
print(summary.loc[list(removed_total)])

Rows before : 1,369,765
Rows after  : 1,263,382
Rows removed: 106,383

{rule: rows_removed}
{'passenger_count_null': 98352, 'pickup_outside_jan_2021': 24, 'dropoff_not_after_pickup': 794, 'fare_amount_le_0': 7213, 'duplicate_rows': 0, 'flag_zero_distance': 0, 'flag_long_distance': 0, 'flag_zero_passengers': 0} 

                          rows_removed  rows_flagged (kept)
passenger_count_null             98352                    0
pickup_outside_jan_2021             24                    0
dropoff_not_after_pickup           794                    0
fare_amount_le_0                  7213                    0
duplicate_rows                       0                    0
flag_zero_distance                   0                17131
flag_long_distance                   0                   17
flag_zero_passengers                 0                26654


**Note on duplicates:** inside `clean()` duplicates are checked per chunk. Step 3 already checked the whole file
with row hashes and found 0 duplicates, so no duplicate is missed by working in chunks.

## Verification
1. Rows before minus every rule's removals must equal rows after.
2. Rows before must match the Postgres count from class (1,369,765).
3. `clean()` must not change the dataframe passed to it.
4. Re-check one cleaned chunk: no rows left that break a drop rule.

In [32]:
assert rows_before == 1_369_765, "row count differs from the class COUNT(*)"
assert rows_before - sum(removed_total.values()) == rows_after, "removed counts do not add up"
assert original_untouched, "clean() modified the original dataframe"

sample_chunk = pd.read_csv(DATA_FILE, nrows=CHUNK_SIZE)
c, _ = clean(sample_chunk)
assert c["passenger_count"].notna().all()
assert c["tpep_pickup_datetime"].between("2021-01-01", "2021-02-01", inclusive="left").all()
assert (c["tpep_dropoff_datetime"] > c["tpep_pickup_datetime"]).all()
assert (c["fare_amount"] > 0).all()
assert not c.duplicated().any()

print("All checks passed")
print(f"{rows_before:,} - {sum(removed_total.values()):,} = {rows_after:,}")
print("\nCleaned dtypes:")
print(c.dtypes)

All checks passed
1,369,765 - 106,383 = 1,263,382

Cleaned dtypes:
VendorID                          Int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   Int64
trip_distance                   float64
RatecodeID                        Int64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      Int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
flag_zero_distance                 bool
flag_long_distance                 bool
flag_zero_passengers               bool
dtype: object


## Step 6 — Stop here
The cleaned data is **not** written to PostgreSQL. Next session `clean()` will be plugged into `ingest_data.py`
between reading each chunk and `to_sql()`.